# Module 01 — Getting familiar with Gymnasium

**Reinforcement Learning · Dakar Institute of Technology (DIT)**

Before we implement any RL algorithm, we need to be comfortable with the
*environment* — the world the agent interacts with. In this course we use
[**Gymnasium**](https://gymnasium.farama.org/) (the maintained successor of
OpenAI Gym), the standard API for RL environments.

### Learning objectives
By the end of this notebook you will be able to:
1. Create an environment and read its **observation** and **action** spaces.
2. Understand the **agent–environment loop**: `reset` → `step` → `reward` → `done`.
3. Run a **random agent** and measure its performance (episode return).
4. **Render** an environment and *watch the agent move* as a GIF.
5. Hand-code a simple **rule-based policy** and see it beat the random agent.

> Everything you learn here (the `reset`/`step` loop) is exactly what your
> learning algorithms in later modules will plug into.

In [ ]:
# --- Course setup: make `rl_helpers` importable from anywhere in the repo ---
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "rl_helpers").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from rl_helpers import set_seed, run_episode, animate_policy

set_seed(0)
print("Gymnasium version:", gym.__version__)

## 1. The agent–environment interface

An RL environment exposes four essential things:

| Concept | Gymnasium call | Meaning |
|---|---|---|
| **Observation space** | `env.observation_space` | what the agent *sees* |
| **Action space** | `env.action_space` | what the agent can *do* |
| **Reset** | `env.reset()` | start a new episode → returns `(obs, info)` |
| **Step** | `env.step(action)` | take an action → `(obs, reward, terminated, truncated, info)` |

Let's inspect the classic **CartPole** environment: balance a pole on a moving cart.

In [ ]:
env = gym.make("CartPole-v1")
print("Observation space:", env.observation_space) # Box(4,) -> cart pos/vel, pole angle/vel
print("Action space: ", env.action_space) # Discrete(2) -> push left / right

obs, info = env.reset(seed=0)
print("\nInitial observation:", obs)
print("info:", info)

### The `step` return value

`env.step(action)` returns **five** things (this is the Gymnasium API — note it
splits the old Gym `done` into two flags):

- `observation` — the next state
- `reward` — scalar feedback for this transition
- `terminated` — `True` if the episode ended *naturally* (goal / failure)
- `truncated` — `True` if it ended due to a **time limit**
- `info` — a dict of diagnostic info (ignore for now)

We usually combine them: `done = terminated or truncated`.

In [ ]:
obs, info = env.reset(seed=0)
action = env.action_space.sample() # a random legal action
obs, reward, terminated, truncated, info = env.step(action)
print("action taken:", action)
print("next obs: ", obs)
print("reward: ", reward)
print("terminated: ", terminated, "| truncated:", truncated)

## 2. A random agent

The simplest possible "policy" is to pick actions uniformly at random. Let's run
one full **episode** and accumulate the **return** (sum of rewards).

> **Return** $G = \sum_{t=0}^{T} r_t$. For CartPole you get `+1` per timestep the
> pole stays up, so the return equals the number of steps survived.

In [ ]:
def random_policy(obs):
    return env.action_space.sample()

# TODO: run ONE episode with random_policy and compute total_reward.
# 1. reset the env (use seed=0)
# 2. loop until the episode is done (terminated OR truncated)
# 3. accumulate the reward into total_reward
obs, info = env.reset(seed=0)
total_reward = 0.0
done = False
steps = 0
while not done:
    action = ... # TODO: choose an action
    # TODO: step the environment and unpack all 5 return values
    # TODO: update total_reward, done, steps
    raise NotImplementedError("Implement the episode loop")

print(f"Random agent survived {steps} steps, return = {total_reward}")

Because the random agent is noisy, evaluate it over **many episodes** and look at
the average — this is how we measure *any* policy in RL.

In [ ]:
def evaluate(policy_fn, n_episodes=50, seed=0):
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        done, G = False, 0.0
        while not done:
            obs, r, term, trunc, _ = env.step(policy_fn(obs))
            G += r
            done = term or trunc
        returns.append(G)
    return np.array(returns)

random_returns = evaluate(random_policy)
print(f"Random policy: mean return = {random_returns.mean():.1f} "
      f"± {random_returns.std():.1f}")

## 3. Watch the agent move

Numbers are abstract — let's *see* the agent. We create the env with
`render_mode="rgb_array"`, roll out an episode capturing frames, and display it as
an animated GIF. The helper `animate_policy` does all of this in one line.

In [ ]:
render_env = gym.make("CartPole-v1", render_mode="rgb_array")
# `animate_policy` rolls out the policy, records frames, and shows the GIF inline.
# NOTE: it uses the env you pass, so `random_policy` must sample from `render_env`.
animate_policy(render_env,
               policy_fn=lambda obs: render_env.action_space.sample(),
               max_steps=200, fps=30, path="cartpole_random.gif")

You should see the pole fall over quickly — the random agent has no idea what it's
doing! Let's fix that with a tiny bit of physics intuition.

## 4. A hand-coded (rule-based) policy

The CartPole observation is `[cart_position, cart_velocity, pole_angle, pole_angular_velocity]`.
A simple, surprisingly effective heuristic: **push in the direction the pole is
falling**. If the pole leans right (`pole_angle > 0`), push the cart right
(`action = 1`); otherwise push left (`action = 0`).

In [ ]:
def heuristic_policy(obs):
    # obs = [cart_position, cart_velocity, pole_angle, pole_angular_velocity]
    # TODO: return action 1 (push right) if the pole leans right, else 0 (push left)
    raise NotImplementedError("Implement the heuristic")

heuristic_returns = evaluate(heuristic_policy)
print(f"Heuristic policy: mean return = {heuristic_returns.mean():.1f} "
      f"± {heuristic_returns.std():.1f}")
print(f"(random was {random_returns.mean():.1f})")

In [ ]:
# Compare the two policies visually
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([random_returns, heuristic_returns], tick_labels=["random", "heuristic"])
ax.set_ylabel("Episode return")
ax.set_title("Random vs. hand-coded policy on CartPole")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# And watch the smarter agent balance for much longer:
animate_policy(render_env, heuristic_policy, max_steps=300, fps=30,
               path="cartpole_heuristic.gif")

## 5. A different world: FrozenLake

Not all environments are continuous. **FrozenLake** is a *discrete* grid world:
the agent (S) must reach the goal (G) without falling into holes (H). The ice is
slippery, so actions are stochastic. This is the kind of environment we'll solve
*exactly* with Dynamic Programming in Module 02.

In [ ]:
lake = gym.make("FrozenLake-v1", is_slippery=True, render_mode="ansi")
obs, _ = lake.reset(seed=0)
print("Observation space:", lake.observation_space, "-> a single integer (which tile)")
print("Action space: ", lake.action_space, "-> 0:Left 1:Down 2:Right 3:Up")
print("\nThe map:")
print(lake.render())

### Quick checks (same environments)

As a warm-up, try answering:
1. What is the *maximum possible* return in `CartPole-v1`? (Hint: check the time limit.)
2. Why does `truncated` exist separately from `terminated`? Give one concrete example.
3. In FrozenLake with `is_slippery=True`, why can the same action lead to different next states?

## Recap

- Gymnasium environments share one API: `reset()` and `step(action)`.
- A **policy** maps observations → actions; we evaluate it by average **return**.
- Even a trivial hand-coded policy can crush a random one.
- We can **render** any environment to *watch* the agent, which we'll do after
  every algorithm we train in this course.

**Next:** Module 02 — computing the *optimal* policy exactly with Dynamic
Programming (policy iteration & value iteration).